In [1]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

from sklearn.model_selection import train_test_split

from sklearn.metrics import *

import tensorflow as tf

import keras

import numpy as np

import matplotlib.pyplot as plt


from tensorflow.keras.utils import to_categorical

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense

from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from glob import glob


import os
import cv2 # computer vision cv2 images would be read


In [2]:
path = r"D:\Imarticus\PSORIASIS skin dataset\PSORIASIS AND NORMAL SKIN"

cate = ['normal_skin' , 'psoriasis']

In [3]:
image_size = 224
input_image = []

for i in cate:
    folders = os.path.join(path, i)
    label = cate.index(i)     # to perform labeling of dog as : 0 and cat as : 1  

    for image in os.listdir(folders):
        image_path = os.path.join(folders , image)
        image_array = cv2.imread(image_path)

        if image_array is None:
            continue  # skip corrupted images
        image_array = cv2.resize(image_array , (image_size , image_size))   # resizing the image 
                                                                            # because in raw data each image size is different and we require 
                                                                            # same size
        input_image.append([image_array , label])

        
        

In [4]:
np.random.shuffle(input_image)

X = []
Y = []

for X_values , labels in input_image:
    X.append(X_values)
    Y.append(labels)



#  Separate X ----> pixels and Y ----> 0 and 1
#  Entire data is stacked on categories
#  1000 images ---> 500 dogs and 500 cats
#  

In [5]:
X = np.array(X)
Y = np.array(Y)

In [6]:
X = X/255

In [29]:
model = tf.keras.models.Sequential() # ----------------------> initialization of the model

model.add(Conv2D(filters=16 , kernel_size=(3,3), activation='relu' , padding='same', input_shape = X.shape[1:]))  # 128 ---> is hyper parameter

model.add(MaxPool2D(pool_size=(2,2)))

model.add(Flatten())

model.add(Dense(128 , activation = 'relu'))   # ----------> Hidden layer 1
model.add(Dense(128 , activation = 'relu'))   # ----------> Hidden layer 2


model.add(Dense(2 , activation='softmax'))

In [30]:
adam = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=adam , loss = 'sparse_categorical_crossentropy' , metrics = ['accuracy'])

model.fit(X, Y, epochs = 10)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 66s 196ms/step - accuracy: 0.8753 - loss: 0.4803 
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 60s 192ms/step - accuracy: 0.9757 - loss: 0.0682
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 61s 194ms/step - accuracy: 0.9883 - loss: 0.0341
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 83s 198ms/step - accuracy: 0.9919 - loss: 0.0226
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 60s 193ms/step - accuracy: 0.9862 - loss: 0.0370
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 193ms/step - accuracy: 0.9940 - loss: 0.0177
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 82s 193ms/step - accuracy: 0.9946 - loss: 0.0173
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 87s 211ms/step - accuracy: 0.9947 - loss: 0.0164
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 63s 200ms/step - accuracy: 0.9918 - loss: 0.0248
Epoch 10/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 62s 198ms/step - accuracy: 0.9924 - loss: 0.0224


In [31]:
pred = model.predict(X)
a = pred.argmax(axis = 1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 41ms/step


In [32]:
print(confusion_matrix(Y , a)) 
print('')
print(classification_report(Y , a))

[[4997    3]
 [   8 4992]]

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5000
           1       1.00      1.00      1.00      5000

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [49]:
len(Y)

9722

# Train Test Split

In [8]:
X_train = X[0 : 7722]
Y_train = Y[0:7722]

X_test = X[7722 :]
Y_test = Y[7722 :]

In [10]:
X_test.shape

(2278, 224, 224, 3)

In [8]:
X_train = np.array(X_train)
Y_train = np.array(Y_train)

X_test = np.array(X_test)

In [9]:
X_train.shape[1:]

(224, 224, 3)

In [10]:

model = tf.keras.models.Sequential() # ----------------------> initialization of the model

model.add(Conv2D(filters=128 , kernel_size=(3,3), activation='relu' , padding='same', input_shape = X_train.shape[1:]))  # 128 ---> is hyper parameter

model.add(MaxPool2D(pool_size=(2,2)))

model.add(Flatten())

model.add(Dense(128 , activation = 'relu'))   # ----------> Hidden layer 1
model.add(Dense(128 , activation = 'relu'))   # ----------> Hidden layer 2


model.add(Dense(2 , activation='softmax'))

In [11]:
adam = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=adam , loss = 'sparse_categorical_crossentropy' , metrics = ['accuracy'])

In [12]:
model.fit(X_train , Y_train , epochs=5 , validation_split=.2, batch_size=32)

Epoch 1/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 354s 2s/step - accuracy: 0.8928 - loss: 1.0849 - val_accuracy: 0.9657 - val_loss: 0.0820
Epoch 2/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 332s 2s/step - accuracy: 0.9650 - loss: 0.0887 - val_accuracy: 0.9508 - val_loss: 0.1333
Epoch 3/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 330s 2s/step - accuracy: 0.9768 - loss: 0.0582 - val_accuracy: 0.9495 - val_loss: 0.1556
Epoch 4/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 383s 2s/step - accuracy: 0.9802 - loss: 0.0533 - val_accuracy: 0.9851 - val_loss: 0.0402
Epoch 5/5
194/194 ━━━━━━━━━━━━━━━━━━━━ 338s 2s/step - accuracy: 0.9885 - loss: 0.0280 - val_accuracy: 0.9780 - val_loss: 0.0489


In [13]:
pred = model.predict(X_test)

72/72 ━━━━━━━━━━━━━━━━━━━━ 19s 253ms/step


In [14]:
pred_classes = pred.argmax(axis = 1)            #  argmax in each row and it gives index of max value

In [15]:
print(confusion_matrix(Y_test , pred_classes)) 
print('')
print(classification_report(Y_test , pred_classes))

[[1130   28]
 [  22 1098]]

              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1158
           1       0.98      0.98      0.98      1120

    accuracy                           0.98      2278
   macro avg       0.98      0.98      0.98      2278
weighted avg       0.98      0.98      0.98      2278



# VGG16 ------> Train Test Split using Transfer Learning -------> VGG16 model

In [7]:
X_train = X[0 : 7722]
Y_train = Y[0:7722]

X_test = X[7722 :]
Y_test = Y[7722 :]

In [8]:
X_train = np.array(X_train)
Y_train = np.array(Y_train)

X_test = np.array(X_test)

In [9]:
Y_test[0]

np.int64(0)

In [10]:
X = preprocess_input(X)

In [11]:
vgg16_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)


for layer in vgg16_base.layers:
    layer.trainable = False

                                        # Custom classifier on top of VGG16
x = vgg16_base.output
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)    # extra dense layer

output = Dense(2, activation='softmax')(x)

model = Model(inputs=vgg16_base.input, outputs=output)


In [12]:
# vgg = VGG16(input_shape=IMAGE_SIZE + [3], weights='imagenet', include_top=False)
# for layer in vgg.layers:
#   layer.trainable = False

In [13]:
adam = tf.keras.optimizers.Adam(learning_rate=0.0001)

model.compile(
    optimizer=adam,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [14]:
history = model.fit(
    X_train,
    Y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=32
)


Epoch 1/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1385s 7s/step - accuracy: 0.9464 - loss: 0.1364 - val_accuracy: 0.9799 - val_loss: 0.0567
Epoch 2/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1392s 7s/step - accuracy: 0.9846 - loss: 0.0446 - val_accuracy: 0.9877 - val_loss: 0.0414
Epoch 3/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1441s 7s/step - accuracy: 0.9906 - loss: 0.0294 - val_accuracy: 0.9903 - val_loss: 0.0283
Epoch 4/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1418s 7s/step - accuracy: 0.9940 - loss: 0.0202 - val_accuracy: 0.9909 - val_loss: 0.0286
Epoch 5/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1402s 7s/step - accuracy: 0.9937 - loss: 0.0178 - val_accuracy: 0.9935 - val_loss: 0.0232
Epoch 6/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1396s 7s/step - accuracy: 0.9968 - loss: 0.0115 - val_accuracy: 0.9929 - val_loss: 0.0210
Epoch 7/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1397s 7s/step - accuracy: 0.9958 - loss: 0.0126 - val_accuracy: 0.9903 - val_loss: 0.0254
Epoch 8/10
194/194 ━━━━━━━━━━━━━━━━━━━━ 1357s 7s/step - accuracy: 0.9971 - loss: 0.0100 - 

In [15]:
pred = model.predict(X_test)
pred_classes = pred.argmax(axis=1)

72/72 ━━━━━━━━━━━━━━━━━━━━ 405s 6s/step


In [16]:
print(confusion_matrix(Y_test, pred_classes))
print()
print(classification_report(Y_test, pred_classes))

[[1164    2]
 [   3 1109]]

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1166
           1       1.00      1.00      1.00      1112

    accuracy                           1.00      2278
   macro avg       1.00      1.00      1.00      2278
weighted avg       1.00      1.00      1.00      2278



In [17]:
model.save("psorasis.h5")

In [ ]:
def predict_single_image(image_path):

    # Class labels (must match training order)
    class_names = ['normal_skin', 'psoriasis']

    # Read image
    img = cv2.imread(image_path)

    if img is None:
        print("Image not found!")
        return

    # Convert BGR → RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize to 224x224
    img = cv2.resize(img, (224, 224))

    # Convert to numpy array
    img = np.array(img)

    # Expand dimensions (model expects batch input)
    img = np.expand_dims(img, axis=0)

    # IMPORTANT: Same preprocessing used during training
    img = preprocess_input(img)

    # Predict
    prediction = model.predict(img)

    predicted_class = np.argmax(prediction)
    confidence = np.max(prediction)

    print("Prediction:", class_names[predicted_class])
    print("Confidence:", round(confidence * 100, 2), "%")

    if predicted_class == 1:
        print("⚠️ This image shows signs of Psoriasis.")
    else:
        print("✅ This image appears to be Normal Skin.")


In [ ]:
predict_single_image(r"D:\test_image.jpg")
